# Chapter 11 · Gateways, registries and A2A

Northside Veterinary Clinic has been connecting services: the lab, grooming, the shop, boarding, insurance, suppliers, staff, accounting and communications. With the clinic's three servers, that makes twelve servers and 56 tools. A gateway gathers them in a single entry point and decides, in each task, which tools the model sees: all of them, or the ones most similar to the task according to an embedding model.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model and the clinic

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0 sentence-transformers  # the versions the book was checked with

In [ ]:
"""Chapter 11: gateways and registries. Twelve servers and 56 tools behind a gateway
that chooses, in each task, the ones the model sees."""
import asyncio
import copy
import inspect
import json
import re
import subprocess
import sys
import unicodedata
import urllib.request
from contextlib import AsyncExitStack
from datetime import date
from pathlib import Path
from typing import Annotated

import torch
from mcp import Client, StdioServerParameters
from mcp.server import MCPServer
from pydantic import Field
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
MODEL = None                               # None: choose according to the hardware
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/en/"
TODAY = date(2026, 10, 20)                 # the date of the clinic's data


def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; if not, on the CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def download(name):
    """A JSON file from the data folder; in Colab, it is downloaded."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
tok, model = load(MODEL)

In [ ]:
def reset():
    """Puts the clinic back as it was: the original schedule and no notices."""
    global APPOINTMENTS, NOTICES
    APPOINTMENTS, NOTICES = copy.deepcopy(APPOINTMENTS0), []


def strip_accents(text):
    """Lowercase and without accents, to compare names however they are written."""
    return "".join(c for c in unicodedata.normalize("NFD", str(text).lower())
                   if unicodedata.category(c) != "Mn").strip()


def book_appointment(pet, vet, date, time, reason):
    """Books a free slot in the schedule; returns the appointment or the problem."""
    if pet not in PETS:
        return {"error": f"There is no pet {pet}."}
    if vet not in VETS:
        return {"error": f"There is no vet {vet}."}
    if any(a["date"] == date and a["time"] == time and a["vet"] == vet
           and a["status"] != "canceled" for a in APPOINTMENTS):
        return {"error": f"{VETS[vet]} already has an appointment on {date} at {time}."}
    appointment = {"id": f"AP-{len(APPOINTMENTS0) + len(APPOINTMENTS) + 1:04d}",
                   "date": date, "time": time, "vet": vet, "pet": pet,
                   "reason": reason, "status": "scheduled"}
    APPOINTMENTS.append(appointment)
    return appointment


def cancel_appointment(appointment):
    """Cancels a scheduled appointment."""
    for a in APPOINTMENTS:
        if a["id"] == appointment:
            if a["status"] != "scheduled":
                return {"error": f"Appointment {appointment} is \"{a['status']}\"."}
            a["status"] = "canceled"
            return {"canceled": appointment, "date": a["date"], "time": a["time"]}
    return {"error": f"There is no appointment {appointment}."}


def notify(owner, text):
    """Sends a notice to the owner (here it is stored in a list)."""
    if owner not in OWNERS:
        return {"error": f"There is no client {owner}."}
    NOTICES.append({"owner": owner, "text": text})
    return {"sent": True, "to": OWNERS[owner]["name"],
            "via": OWNERS[owner]["notify_by"]}


def age(birth_date):
    """Full years on the date of the data."""
    b = date.fromisoformat(birth_date)
    return TODAY.year - b.year - ((TODAY.month, TODAY.day) < (b.month, b.day))


def pet_summary(pid):
    """The essentials of a pet, without the full history."""
    p = PETS[pid]
    return {"id": pid, "name": p["name"], "species": p["species"], "breed": p["breed"],
            "age_years": age(p["birth_date"]), "current_weight_kg": p["weight_kg"],
            "allergies": p["allergies"] or "none known",
            "owner": {"id": p["owner"], "name": OWNERS[p["owner"]]["name"]}}


def vaccines_with_status(pid):
    """Each vaccine with its next date and whether it is already overdue."""
    return [{"vaccine": v["vaccine"], "next": v["next"],
             "status": "overdue" if v["next"] < TODAY.isoformat() else "up to date"}
            for v in VACCINES[pid]]


def client_record_3(client_id):
    """A client's record with their pets and unpaid invoices (chapter 1, toolset 3)."""
    if client_id not in OWNERS:
        return {"error": f"There is no client {client_id}. Look them up by name with "
                         "search(table='owners', field='name', value=...)."}
    unpaid = [i for i in INVOICES if i["owner"] == client_id and not i["paid"]]
    return {"id": client_id, "name": OWNERS[client_id]["name"],
            "notify_by": OWNERS[client_id]["notify_by"],
            "pets": [{"id": p, "name": d["name"], "species": d["species"]}
                     for p, d in PETS.items() if d["owner"] == client_id],
            "unpaid_invoices": [{"id": i["id"], "date": i["date"], "total": i["total"]}
                                for i in unpaid],
            "total_unpaid": round(sum(i["total"] for i in unpaid), 2)}


def schedule_3(date):
    """A day's appointments and each vet's free times (chapter 1, toolset 3)."""
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(date)):
        return {"error": f"The date \"{date}\" is not valid; use YYYY-MM-DD."}
    appointments = [a for a in APPOINTMENTS
                    if a["date"] == date and a["status"] != "canceled"]
    return {"date": date,
            "appointments": [f"{a['id']} {a['time']} {a['vet']} "
                             f"{PETS[a['pet']]['name']} ({a['pet']})"
                             for a in appointments],
            "vets": VETS,
            "free_times": {v: [t for t in TIMES if not any(
                a["time"] == t and a["vet"] == v for a in appointments)]
                for v in VETS}}


def locate(pet, owner=""):
    """The pet's id from its name (and the owner's, if there are several)."""
    ids = [p for p, d in PETS.items() if strip_accents(pet) == strip_accents(d["name"])
           and strip_accents(owner) in strip_accents(OWNERS[d["owner"]]["name"])]
    if not ids:
        return None, {"error": f"There is no pet called \"{pet}\""
                               + (f" belonging to {owner}" if owner else "")
                               + " at the clinic."}
    if len(ids) > 1:
        return None, {"error": f"There are {len(ids)} pets called \"{pet}\"; give the "
                               "owner's name.",
                      "pets": [pet_summary(p) for p in ids]}
    return ids[0], None


def vet_of(name):
    """The vet's id from their name or their id."""
    for v, n in VETS.items():
        if strip_accents(name) in (strip_accents(v), strip_accents(n)) or \
                strip_accents(name) in strip_accents(n):
            return v
    return None


def find_pet(name, owner=""):
    """A pet's data, looked up by its name."""
    pid, error = locate(name, owner)
    return error or pet_summary(pid)


def pets_of(owner):
    """A client's pets, the client looked up by name."""
    ids = [c for c, x in OWNERS.items()
           if strip_accents(owner) in strip_accents(x["name"])]
    if not ids:
        return {"error": f"There is no client called \"{owner}\"."}
    return {"clients": [{"name": OWNERS[c]["name"], "pets": [
        {"name": x["name"], "species": x["species"], "breed": x["breed"]}
        for p, x in PETS.items() if x["owner"] == c]} for c in ids]}


def vaccines(pet, owner=""):
    """A pet's vaccines, with their status."""
    pid, error = locate(pet, owner)
    return error or {"pet": PETS[pid]["name"], "today": TODAY.isoformat(),
                     "vaccines": vaccines_with_status(pid)}


def history(pet, last=3, owner=""):
    """A pet's last visits."""
    pid, error = locate(pet, owner)
    if error:
        return error
    return {"pet": PETS[pid]["name"], "visits": [
        {k: v[k] for k in ("date", "reason", "treatment", "weight_kg")}
        for v in HISTORY[pid][-int(last):]]}


def dose(medicine, pet, owner=""):
    """A medicine's dose for a pet, with allergy and species warnings."""
    med = PHARMACY.get(strip_accents(medicine))
    if med is None:
        return {"error": f"We don't have \"{medicine}\". Medicines: "
                         + ", ".join(PHARMACY) + "."}
    pid, error = locate(pet, owner)
    if error:
        return error
    p = PETS[pid]
    warnings = []
    if strip_accents(medicine) in [strip_accents(a) for a in p["allergies"]]:
        warnings.append(f"{p['name']} is allergic to {medicine}: do not prescribe.")
    if p["species"] not in med["species"]:
        warnings.append(f"{medicine} is not indicated for {p['species']}s.")
    return {"medicine": strip_accents(medicine), "pet": p["name"],
            "weight_kg": p["weight_kg"], "dose_mg_per_kg": med["dose_mg_kg"],
            "dose_mg_per_dose": round(med["dose_mg_kg"] * p["weight_kg"], 2),
            "stock": med["stock"],
            "restock": med.get("restock", "no date") if not med["stock"] else None,
            "warnings": warnings or "none"}


def free_slots(date, vet=""):
    """A day's free times, for one vet or for all."""
    r = schedule_3(date)
    if "error" in r:
        return r
    vets = [vet_of(vet)] if vet else list(VETS)
    if vets == [None]:
        return {"error": f"There is no vet \"{vet}\". They are: "
                         + ", ".join(VETS.values()) + "."}
    return {"date": date, "free_times": {VETS[v]: r["free_times"][v] for v in vets}}


def appointments_of(pet, owner=""):
    """A pet's appointments, past and scheduled, with their ids."""
    pid, error = locate(pet, owner)
    return error or {"pet": PETS[pid]["name"], "today": TODAY.isoformat(),
                     "appointments": [
        {"id": a["id"], "date": a["date"], "time": a["time"], "reason": a["reason"],
         "vet": VETS[a["vet"]], "status": a["status"]}
        for a in APPOINTMENTS if a["pet"] == pid]}


def accounts(owner):
    """A client's unpaid invoices and the total owed."""
    ids = [c for c, x in OWNERS.items()
           if strip_accents(owner) in strip_accents(x["name"])]
    if not ids:
        return {"error": f"There is no client called \"{owner}\"."}
    return {"clients": [{k: v for k, v in client_record_3(c).items() if k != "id"}
                        for c in ids]}


def book(pet, vet, date, time, reason, owner=""):
    """Books an appointment from the pet's and the vet's names."""
    pid, error = locate(pet, owner)
    if error:
        return error
    v = vet_of(vet)
    if v is None:
        return {"error": f"There is no vet \"{vet}\". They are: "
                         + ", ".join(VETS.values()) + "."}
    r = book_appointment(pid, v, date, time, reason)
    if "error" in r:
        return r
    return {"booked": r["id"], "pet": PETS[pid]["name"], "vet": VETS[v],
            "date": date, "time": time}


def notify_owner_of(pet, text, owner=""):
    """Sends a notice to the owner of a pet."""
    pid, error = locate(pet, owner)
    return error or notify(PETS[pid]["owner"], text)


TASKS_SET = {"find_pet": find_pet, "pets_of": pets_of, "vaccines": vaccines,
             "history": history, "dose": dose, "free_slots": free_slots,
             "appointments_of": appointments_of, "accounts": accounts,
             "book_appointment": book, "cancel_appointment": cancel_appointment,
             "notify_owner": notify_owner_of}

In [ ]:
def tool(name, description, parameters):
    """A tool's schema; the parameters without a default value are required."""
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object",
                       "required": [p for p, (_, req) in parameters.items() if req],
                       "properties": {p: {"type": "string", "description": d}
                                      for p, (d, _) in parameters.items()}}}}


PET = ("Name of the pet.", True)


OWNER = ("Owner's name; only if several pets have that name.", False)


SCHEMAS_TASKS = [
    tool("find_pet", "A pet's data, by name: species, age, current weight, allergies "
         "and owner.", {"name": PET, "owner": OWNER}),
    tool("pets_of", "A client's pets, the client looked up by name.",
         {"owner": ("Owner's name.", True)}),
    tool("vaccines", "A pet's vaccines: the next date of each one and whether it is "
         "overdue.", {"pet": PET, "owner": OWNER}),
    tool("history", "A pet's last visits: date, reason, treatment and weight.",
         {"pet": PET, "last": ("How many visits (3 by default).", False),
          "owner": OWNER}),
    tool("dose", "Dose of a medicine for a pet according to its current weight, with "
         "allergy or species warnings and the stock.",
         {"medicine": ("Name of the medicine.", True), "pet": PET, "owner": OWNER}),
    tool("free_slots", "Free times in the schedule on a day, for one vet or for all.",
         {"date": ("YYYY-MM-DD.", True),
          "vet": ("Name of the vet; empty for all.", False)}),
    tool("appointments_of", "A pet's appointments (past and scheduled), with their id.",
         {"pet": PET, "owner": OWNER}),
    tool("accounts", "A client's unpaid invoices, the client looked up by name, and "
         "the total owed.", {"owner": ("Owner's name.", True)}),
    tool("book_appointment", "Books an appointment at a free time.",
         {"pet": PET, "vet": ("Name of the vet.", True), "date": ("YYYY-MM-DD.", True),
          "time": ("HH:MM, on the hour or half past.", True),
          "reason": ("Reason for the appointment.", True), "owner": OWNER}),
    tool("cancel_appointment", "Cancels a scheduled appointment.",
         {"appointment": ("Id of the appointment, like AP-0026 (given by "
                          "appointments_of).", True)}),
    tool("notify_owner", "Sends a notice to a pet's owner, by email or SMS as they "
         "prefer.", {"pet": PET, "text": ("Text of the notice, in English.", True),
                     "owner": OWNER}),
]


SYSTEM = (
    "You are the assistant of the staff of Northside Veterinary Clinic, a veterinary "
    "clinic in Portland, Oregon. Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. Call only one tool at a time, and "
    "before each tool call write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. When the task is finished, answer "
    "in English, in one or two sentences.")


def generate(messages, schemas, max_tokens=400):
    """One answer from the model, and the input and output tokens."""
    text = tok.apply_chat_template(messages, tools=schemas, tokenize=False,
                                   add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    return (tok.decode(new, skip_special_tokens=True).strip(),
            inputs["input_ids"].shape[1], len(new))


def read_calls(answer):
    """The calls the model asks for, written between <tool_call> and </tool_call>."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", answer, re.S):
        try:
            call = json.loads(block)
            calls.append({"name": call["name"], "arguments": call.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            calls.append({"name": None, "arguments": {}})
    return calls


def execute(call, functions):
    """Runs a call if the tool exists and the arguments fit."""
    function = functions.get(call["name"])
    if function is None:
        return {"error": f"There is no tool {call['name']}."}
    try:
        return function(**call["arguments"])
    except (TypeError, ValueError, KeyError) as e:
        return {"error": f"Invalid arguments: {e}"}


def agent(task, functions, schemas, max_steps=8, show=False):
    """Think, act, observe and repeat. It counts the tokens of the results."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        answer, n_in, n_out = generate(messages, schemas)
        calls = read_calls(answer)[:1]                 # one at a time
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_in, "output_tokens": n_out,
                      "thought": thought, "calls": []})
        if not calls:
            trace[-1]["answer"] = answer
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        if show and thought:
            print(f"[{step}] {thought}")
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            result = execute(call, functions)
            text = json.dumps(result, ensure_ascii=False)
            trace[-1]["calls"].append({**call, "result": result,
                                       "result_tokens": len(tok(text)["input_ids"])})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": text})
            if show:
                args = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({args})")
                print(f"     → {text[:150]}")
    return None, trace


def says(answer, *forms):
    """Does any of those forms appear in the answer?"""
    return any(re.search(f, answer or "", re.I) for f in forms)


MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August",
          "September", "October", "November", "December"]


def date_in(answer, iso):
    """Does the date appear, as 2027-08-22, 8/22/2027, August 22, 2027 or 22 August?"""
    y, m, d = iso.split("-")
    month = MONTHS[int(m) - 1][:3]
    return says(answer, iso, rf"\b0?{int(m)}/0?{int(d)}/{y}",
                rf"\b{month}[a-z]*\.? 0?{int(d)}(st|nd|rd|th)?\b",
                rf"\b0?{int(d)}(st|nd|rd|th)? (of )?{month}")


def time_in(answer, hhmm):
    """Does the time appear, as 16:30, 4:30 p.m., 4:30pm or 4:30 (4 p.m. for 16:00)?"""
    h, m = (int(x) for x in hhmm.split(":"))
    h12, right, wrong = (h - 1) % 12 + 1, "pa"[h < 12], "ap"[h < 12]
    marker, not_wrong = rf"\s*{right}\.?\s?m\b", rf"(?!\s*{wrong}\.?\s?m\b)"
    forms = [rf"\b0?{h}:{m:02d}\b", rf"\b{h12}:{m:02d}\b{not_wrong}"]
    if m == 0:
        forms += [rf"\b{h12}{marker}",
                  rf"\b{h12}\b{not_wrong}(?=\s*(,|and\b|or\b|o'clock))"]
    if (h, m) == (12, 0):
        forms.append(r"\bnoon\b")
    return says(answer, *forms)


def new_appointments():
    """The appointments created during the task."""
    return [a for a in APPOINTMENTS if a["id"] not in {x["id"] for x in APPOINTMENTS0}]


def first_free(date, vet):
    """The first free time of a vet on a day, in the original schedule."""
    taken = {a["time"] for a in APPOINTMENTS0 if a["date"] == date and a["vet"] == vet}
    return next(t for t in TIMES if t not in taken)


def owes(name):
    """What a client owes in unpaid invoices."""
    return round(sum(i["total"] for i in INVOICES if not i["paid"]
                     and OWNERS[i["owner"]]["name"] == name), 2)


def amount_in(answer, value):
    """Does the amount appear, as 136, 136.00, $136 or 136 dollars?"""
    whole = f"{value:.2f}".replace(".00", "").replace(".", r"\.")
    return says(answer, rf"\b{whole}(\.00?)?\b")


def notice_to(name, *forms):
    """Was a notice sent to that client, with all those forms in its text?"""
    return any(OWNERS[n["owner"]]["name"] == name
               and all(re.search(f, n["text"], re.I) for f in forms) for n in NOTICES)


TASKS = [
    ("rabies_vaccine", "When is Lola, Martha Fisher's dog, due for her next rabies "
     "vaccine?",
     lambda r: date_in(r, "2027-08-22")),
    ("dose", "How many milligrams of meloxicam should Max get per dose?",
     lambda r: says(r, r"2\.07", r"2\.1\b")),
    ("allergy", "They're going to prescribe amoxicillin for Simba. Is there any "
     "problem?",
     lambda r: says(r, r"allerg")),
    ("book", "Book a checkup for Toby with Dr. Paul Meyer on Thursday the 22nd, at the "
     "first free time.",
     lambda r: [(a["pet"], a["vet"], a["date"], a["time"]) for a in new_appointments()]
     == [("P-002", "V2", "2026-10-22", first_free("2026-10-22", "V2"))]),
    ("cancel", "Cancel Rocky's appointment tomorrow.",
     lambda r: [a["id"] for a in APPOINTMENTS if a["status"] == "canceled"]
     == ["AP-0060"]),
    ("overdue_notice", "Let Toby's owner know about his overdue vaccines.",
     lambda r: notice_to("Peter Adams", "rabies", "DHPP", "leptospirosis")),
    ("debt", "How much does Oscar Price owe in unpaid invoices?",
     lambda r: amount_in(r, owes("Oscar Price"))),
    ("marthas_cat", "Martha Fisher is calling about her cat. Is any of her vaccines "
     "overdue?",
     lambda r: says(r, "FVRCP")
     and says(r, "overdue|expired|past due|out of date|lapsed")),
    ("weight", "How much did Zeus weigh at his last visit, and how much has it changed "
     "since the one before?",
     lambda r: says(r, r"32\.2") and says(r, r"0\.6")
     and not says(r, r"(last|latest|most recent) visit[^.,]{0,30}(2025|31\.6)",
                  r"31\.6 kg (at|on|in|during) (his|the) (last|latest|most recent)")),
    ("last_visit", "Why did Willow come in last time, and what treatment did she get?",
     lambda r: says(r, "wound") and says(r, "amoxicillin")),
    ("free_slots", "What times is Dr. Anna Sutton free on Wednesday the 21st in the "
     "afternoon?",
     lambda r: all(time_in(r, t) for t in ("16:00", "16:30", "17:30", "18:00", "18:30"))
     and not time_in(r, "17:00")),
    ("does_not_exist", "When is Buster due for his rabies vaccine?",
     lambda r: says(r, r"\bno (pet|patient|record|animal|such|one|information)\b",
                    r"(n't|not|unable to) (exist|find|locate|appear|registered)",
                    r"\bnot (found|a patient|registered|on file|in our)",
                    r"\bthere (is|are) no\b", r"\bnone\b")
     and not says(r, r"\b20\d\d-\d\d-\d\d\b", r"\b\d\d?/\d\d?/20\d\d\b")),
]

## 2. The clinic's three servers (those of chapter 4)

In [ ]:
%%writefile pharmacy_server.py
"""MCP server of the clinic's pharmacy: prices and stock.

It is a separate program, as if another team had written it: it shares no code with the
agent. It talks through standard input and output (stdio), with JSON-RPC messages.
Usage: python pharmacy_server.py [data_folder]
"""
import json
import re
import sys
import unicodedata
from pathlib import Path
from typing import Annotated

from mcp.server import MCPServer
from pydantic import Field

DATA = Path(sys.argv[1] if len(sys.argv) > 1 else "data")
PHARMACY = json.loads((DATA / "pharmacy.json").read_text(encoding="utf-8"))
PRICES = json.loads((DATA / "prices.json").read_text(encoding="utf-8"))

pharmacy = MCPServer("pharmacy", instructions="Prices and stock of the clinic.")


def roots(text):
    """The first seven letters of each word, without accents, to compare names."""
    text = "".join(c for c in unicodedata.normalize("NFD", text.lower())
                   if unicodedata.category(c) != "Mn")
    return {w[:7] for w in re.findall(r"\w{4,}", text)}


@pharmacy.tool()
def price(service: Annotated[str, Field(description="Service or medicine.")]) -> dict:
    """Price in dollars of a service (consultation, vaccines, tests…) or a medicine."""
    prices = {**PRICES, **{k: d["price"] for k, d in PHARMACY.items()}}
    shared = {k: len(roots(service) & roots(k)) for k in prices}
    best = max(shared.values())
    if best == 0:
        return {"error": f"There is no price for \"{service}\". Services: "
                         + ", ".join(PRICES) + "; and the pharmacy's medicines."}
    return {"prices_usd": {k: prices[k] for k, c in shared.items() if c == best}}


@pharmacy.tool()
def stock(medicine: Annotated[str, Field(description="Name of the medicine.")]) -> dict:
    """Units of a medicine in the pharmacy and, if none are left, when they arrive."""
    key = next((k for k in PHARMACY if roots(k) & roots(medicine)), None)
    if key is None:
        return {"error": f"We don't have \"{medicine}\". Medicines: "
                         + ", ".join(PHARMACY) + "."}
    m = PHARMACY[key]
    return {"medicine": key, "presentation": m["presentation"], "stock": m["stock"],
            "restock": m.get("restock", "no date") if not m["stock"] else None}


if __name__ == "__main__":
    pharmacy.run()          # by default, stdio

In [ ]:
Pet = Annotated[str, Field(description="Name of the pet.")]


Owner = Annotated[str, Field(description="Owner's name; only if several pets have "
                                         "that name.")]


Date = Annotated[str, Field(description="YYYY-MM-DD.")]


clinic = MCPServer("clinic", instructions="Pets, vaccines, medical history, schedule "
                   "and notices of Northside Veterinary Clinic.")


@clinic.tool(name="find_pet")
def mcp_find_pet(name: Pet, owner: Owner = "") -> dict:
    """A pet's data, by name: species, age, current weight, allergies and owner."""
    return find_pet(name, owner)


@clinic.tool(name="pets_of")
def mcp_pets_of(owner: Annotated[str, Field(description="Owner's name.")]) -> dict:
    """A client's pets, the client looked up by name."""
    return pets_of(owner)


@clinic.tool(name="vaccines")
def mcp_vaccines(pet: Pet, owner: Owner = "") -> dict:
    """A pet's vaccines: the next date of each one and whether it is overdue."""
    return vaccines(pet, owner)


@clinic.tool(name="history")
def mcp_history(pet: Pet,
                last: Annotated[int, Field(description="How many visits.")] = 3,
                owner: Owner = "") -> dict:
    """A pet's last visits: date, reason, treatment and weight."""
    return history(pet, last, owner)


@clinic.tool(name="dose")
def mcp_dose(medicine: Annotated[str, Field(description="Name of the medicine.")],
             pet: Pet, owner: Owner = "") -> dict:
    """Dose of a medicine for the current weight, with allergy or species warnings."""
    return dose(medicine, pet, owner)


@clinic.tool(name="free_slots")
def mcp_free_slots(date: Date, vet: Annotated[str, Field(
        description="Name of the vet; empty for all.")] = "") -> dict:
    """Free times in the schedule on a day, for one vet or for all."""
    return free_slots(date, vet)


@clinic.tool(name="appointments_of")
def mcp_appointments_of(pet: Pet, owner: Owner = "") -> dict:
    """A pet's appointments (past and scheduled), with their id."""
    return appointments_of(pet, owner)


@clinic.tool(name="book_appointment")
def mcp_book_appointment(pet: Pet,
                         vet: Annotated[str, Field(description="Name of the vet.")],
                         date: Date,
                         time: Annotated[str, Field(
                             description="HH:MM, on the hour or half past.")],
                         reason: Annotated[str, Field(
                             description="Reason for the appointment.")],
                         owner: Owner = "") -> dict:
    """Books an appointment at a free time."""
    return book(pet, vet, date, time, reason, owner)


@clinic.tool(name="cancel_appointment")
def mcp_cancel_appointment(appointment: Annotated[str, Field(
        description="Id of the appointment, like AP-0026 (given by "
                    "appointments_of).")]) -> dict:
    """Cancels a scheduled appointment."""
    return cancel_appointment(appointment)


@clinic.tool(name="notify_owner")
def mcp_notify_owner(pet: Pet,
                     text: Annotated[str, Field(
                         description="Text of the notice, in English.")],
                     owner: Owner = "") -> dict:
    """Sends a notice to a pet's owner, by email or SMS as they prefer."""
    return notify_owner_of(pet, text, owner)


# The pharmacy is another program, pharmacy_server.py; it is launched as a separate
# process, and we talk to it through its standard input and output (stdio).
PHARMACY_SERVER = Path("pharmacy_server.py")      # in the notebook, in the working folder


pharmacy = StdioServerParameters(command=sys.executable,
                                 args=[str(PHARMACY_SERVER), str(DATA)])


billing = MCPServer("billing", instructions="Invoices of the clinic's clients.")


@billing.tool(name="accounts")
def mcp_accounts(owner: Annotated[str, Field(description="Owner's name.")]) -> dict:
    """A client's unpaid invoices, the client looked up by name, and the total owed."""
    return accounts(owner)

In [ ]:
def without_titles(schema):
    """The schema without the "title" keys that Pydantic adds (the model needs none)."""
    if isinstance(schema, dict):
        return {k: without_titles(v) for k, v in schema.items() if k != "title"}
    return schema


def compact(text):
    """A result's JSON on a single line, without the indentation the server sends."""
    try:
        return json.dumps(json.loads(text), ensure_ascii=False)
    except ValueError:
        return text


async def mcp_agent(task, host, max_steps=8, show=False):
    """Chapter 1's loop, with the tools of the MCP servers."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        answer, n_in, n_out = generate(messages, host.schemas)
        calls = read_calls(answer)[:1]
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_in, "thought": thought,
                      "calls": []})
        if not calls:
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            text = await host.call(call["name"], call["arguments"])
            trace[-1]["calls"].append({**call, "result": text,
                                       "result_tokens": len(tok(text)["input_ids"])})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": text})
            if show:
                args = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({args})")
                print(f"     → {text[:150]}")
    return None, trace


def number_in(answer, value):
    """Does the figure appear, with one or two decimals?"""
    return says(answer, *{f"{value:.2f}".replace(".", r"\."),
                          f"{value:.1f}".replace(".", r"\.") + r"\b",
                          rf"\b{value:g}\b".replace(".", r"\.")})


# Ways of saying that none is left; a bare "there is no" is not enough, because
# "there is no such tool" does not say it.
OUT_OF_STOCK = ("out of stock", r"\bsold out", r"not (currently )?available",
                r"is ?n[o']t available", r"\b(0|zero) units",
                r"\b(none|no units|no doxycycline) (is |are )?left",
                r"(don't|do not) have (any|it|doxycycline)( in stock)?",
                r"\bno (stock|units)\b", r"stock (is|of|at) (zero|0)\b")


PHARMACY_TASKS = [
    ("doxycycline", "How much is doxycycline, and do we have it right now?",
     lambda r: number_in(r, 14.2) and says(r, *OUT_OF_STOCK)),
    ("when_it_arrives", "We've run out of doxycycline. When does it arrive?",
     lambda r: date_in(r, PHARMACY["doxycycline"]["restock"])),
    ("meloxicam", "How many units of meloxicam do we have left?",
     lambda r: says(r, rf"\b{PHARMACY['meloxicam']['stock']}\b")),
    ("tobys_vaccines", "How much would it cost to give Toby all his overdue vaccines?",
     lambda r: amount_in(r, sum(PRICES[v["vaccine"] + " vaccine"]
                                for v in vaccines("Toby")["vaccines"]
                                if v["status"] == "overdue"))),
]


BANK = TASKS + PHARMACY_TASKS


def load_data():
    """Load the clinic's data and leave the clinic as it was at the start."""
    global PETS, OWNERS, HISTORY, VACCINES, APPOINTMENTS0, PHARMACY, INVOICES
    global VETS, TIMES, PRICES
    PETS, OWNERS = download("pets.json"), download("owners.json")
    HISTORY, VACCINES = download("history.json"), download("vaccines.json")
    APPOINTMENTS0, PHARMACY = download("appointments.json"), download("pharmacy.json")
    INVOICES, VETS = download("invoices.json"), download("vets.json")
    PRICES = download("prices.json")
    TIMES = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    reset()

In [ ]:
load_data()

## 3. The other nine services

Their tools always give the same answer: what is measured is whether the model chooses well.

In [ ]:
OTHER_SERVICES = {
    "lab": [
        ("order_test", "Order a test for a pet from the outside lab.",
         {"pet": "Name of the pet.", "test": "Test being ordered."}),
        ("test_results", "Results of a pet's lab tests.",
         {"pet": "Name of the pet."}),
        ("sample_status", "Say where a sample sent to the lab is in the process.",
         {"sample": "Code of the sample."}),
        ("lab_prices", "Prices of the outside lab's tests.", {}),
    ],
    "grooming": [
        ("grooming_appointments", "A pet's dog grooming appointments.",
         {"pet": "Name of the pet."}),
        ("book_grooming", "Book a bath or a haircut at the grooming salon.",
         {"pet": "Name of the pet.", "date": "YYYY-MM-DD.",
          "service": "Bath or haircut."}),
        ("cancel_grooming", "Cancel a grooming appointment.",
         {"appointment": "Id of the grooming appointment."}),
        ("grooming_prices", "Grooming prices by the size of the dog.", {}),
        ("grooming_history", "Grooming services a pet has had.",
         {"pet": "Name of the pet."}),
    ],
    "shop": [
        ("find_product", "Look up a product in the shop: pet food, toys, leashes.",
         {"name": "Name of the product."}),
        ("shop_price", "Price of a product in the shop.",
         {"product": "Name of the product."}),
        ("shop_stock", "Units of a product in the shop.",
         {"product": "Name of the product."}),
        ("shop_order", "Place a shop order for a client.",
         {"client": "Name of the client.", "product": "Product.", "units": "Units."}),
        ("shop_deals", "The shop's deals of the week.", {}),
    ],
    "boarding": [
        ("boarding_vacancies", "Free places at the boarding kennel on some dates.",
         {"start_date": "YYYY-MM-DD.", "end_date": "YYYY-MM-DD."}),
        ("book_boarding", "Book a stay at the boarding kennel.",
         {"pet": "Name of the pet.", "start_date": "YYYY-MM-DD.",
          "end_date": "YYYY-MM-DD."}),
        ("cancel_boarding", "Cancel a boarding reservation.",
         {"reservation": "Id of the reservation."}),
        ("boarding_requirements",
         "Vaccines and documents the boarding kennel requires.", {}),
    ],
    "insurance": [
        ("pet_policy", "Data of a pet's insurance policy.",
         {"pet": "Name of the pet."}),
        ("open_claim", "Open an insurance claim for a visit or an operation.",
         {"pet": "Name of the pet.", "reason": "Reason for the claim."}),
        ("claim_status", "Status of a claim open with the insurer.",
         {"claim": "Id of the claim."}),
        ("coverage", "What each plan of the pet insurance covers.", {}),
        ("insurance_history", "A pet's insurance claims and payments.",
         {"pet": "Name of the pet."}),
    ],
    "suppliers": [
        ("supplier_order",
         "Place an order for medicines or supplies with the supplier.",
         {"product": "Product.", "units": "Units."}),
        ("supplier_order_status", "Where an order to the supplier is in the process.",
         {"order": "Id of the order."}),
        ("delivery_receipts", "The supplier's delivery receipts for a month.",
         {"month": "YYYY-MM."}),
        ("supplier_catalog",
         "Products in the supplier's catalog with their cost price.", {}),
    ],
    "staff": [
        ("shifts", "Work shifts of the clinic's staff in a week.",
         {"week": "YYYY-MM-DD of the Monday."}),
        ("time_off", "An employee's approved time off.",
         {"employee": "Name of the employee."}),
        ("on_call", "Vet on call in a week.", {"week": "YYYY-MM-DD of the Monday."}),
        ("timesheets", "Hours clocked by an employee in a month.",
         {"employee": "Name.", "month": "YYYY-MM."}),
        ("training", "The staff's training courses.", {}),
    ],
    "accounting": [
        ("monthly_income", "The clinic's income in a month.", {"month": "YYYY-MM."}),
        ("monthly_expenses", "The clinic's expenses in a month.",
         {"month": "YYYY-MM."}),
        ("quarterly_taxes", "Taxes to be filed for a quarter.",
         {"quarter": "Such as 2026-Q3."}),
        ("supplier_invoices", "Unpaid invoices received from the suppliers.", {}),
        ("cash_closing", "The cash register closing of a day.",
         {"date": "YYYY-MM-DD."}),
    ],
    "communications": [
        ("send_newsletter", "Send the monthly newsletter to all the clients.",
         {"subject": "Subject.", "text": "Text."}),
        ("satisfaction_survey", "Send a satisfaction survey to a client.",
         {"client": "Name of the client."}),
        ("publish_news", "Publish a news item on the clinic's website.",
         {"title": "Title.", "text": "Text."}),
        ("reviews", "The clinic's latest online reviews.", {}),
        ("vaccination_campaign", "Clients to send the spring vaccination campaign to.",
         {}),
        ("sms_reminder", "Schedule a reminder text message for a phone number.",
         {"phone": "Phone number.", "text": "Text."}),
    ],
}


def fixed_tool(service, name, parameters):
    """A function that only answers that the service has received the request. Its
    signature, which is where the SDK gets the schema from, is built with
    inspect.Signature."""
    def function(**arguments):
        return {"service": service, "received": arguments}
    function.__name__ = name
    function.__signature__ = inspect.Signature([
        inspect.Parameter(p, inspect.Parameter.KEYWORD_ONLY,
                          annotation=Annotated[str, Field(description=d)])
        for p, d in parameters.items()])
    return function


def create_service(name, tools):
    """An MCP server with the tools of a service."""
    server = MCPServer(name)
    for tool_name, description, parameters in tools:
        server.add_tool(fixed_tool(name, tool_name, parameters),
                        description=description)
    return server

## 4. The gateway

In [ ]:
EMBEDDER = "intfloat/multilingual-e5-small"


class Gateway:
    """Gathers the tools of all the servers with the server's name in front. With
    selection=k, in each task it gives the model only the k tools whose description is
    most similar to the task, according to an embedding model."""

    def __init__(self, servers, selection=None):
        self.servers, self.selection = servers, selection
        self.schemas = []

    async def __aenter__(self):
        self.stack = AsyncExitStack()
        self.route, self.all = {}, []
        for server in self.servers:
            client = await self.stack.enter_async_context(Client(server))
            for t in (await client.list_tools()).tools:
                name = f"{client.server_info.name}_{t.name}"
                self.route[name] = (client, t.name)
                self.all.append({"type": "function", "function": {
                    "name": name, "description": t.description,
                    "parameters": without_titles(t.input_schema)}})
        self.schemas = self.all
        if self.selection:
            texts = [f"passage: {s['function']['name'].replace('_', ' ')}: "
                     f"{s['function']['description']}" for s in self.all]
            self.vectors = EMBEDDINGS.encode(texts, normalize_embeddings=True,
                                             show_progress_bar=False)
        return self

    async def __aexit__(self, *error):
        await self.stack.aclose()

    def choose(self, task):
        """The tools the model will see in this task: all, or the k most similar."""
        if self.selection:
            query = EMBEDDINGS.encode([f"query: {task}"], normalize_embeddings=True,
                                      show_progress_bar=False)[0]
            order = (self.vectors @ query).argsort()[::-1][:self.selection]
            self.schemas = [self.all[i] for i in sorted(order)]
        return [s["function"]["name"] for s in self.schemas]

    async def call(self, name, arguments):
        """Send the call to the server that offers the tool, with its original name."""
        if name not in self.route:
            return json.dumps({"error": f"There is no tool {name}."},
                              ensure_ascii=False)
        client, original = self.route[name]
        r = await client.call_tool(original, arguments)
        return compact("\n".join(b.text for b in r.content if b.type == "text"))

In [ ]:
EMBEDDINGS = SentenceTransformer(EMBEDDER, device="cpu")     # on the CPU: the GPU is the language model's

## 5. Sixteen tasks and four variants

In [ ]:
NEEDS = {
    "rabies_vaccine": [{"vaccines"}], "dose": [{"dose"}],
    "allergy": [{"find_pet"}, {"dose"}], "book": [{"book_appointment"}],
    "cancel": [{"appointments_of", "cancel_appointment"}],
    "overdue_notice": [{"vaccines", "notify_owner"}], "debt": [{"accounts"}],
    "marthas_cat": [{"pets_of", "vaccines"}], "weight": [{"history"}],
    "last_visit": [{"history"}], "free_slots": [{"free_slots"}],
    "does_not_exist": [{"vaccines"}, {"find_pet"}], "doxycycline": [{"price", "stock"}],
    "when_it_arrives": [{"stock"}], "meloxicam": [{"stock"}],
    "tobys_vaccines": [{"vaccines", "price"}],
}


def found(key, names):
    """Is any of the sets of tools that solve the task among the chosen ones?"""
    short = {n.split("_", 1)[1] for n in names}
    return any(needed <= short for needed in NEEDS[key])


async def solve(servers, selection=None, tasks=BANK):
    """The sixteen tasks behind a gateway, with all the tools or a selection."""
    rows = []
    async with Gateway(servers, selection) as gateway:
        for key, task, check in tasks:
            reset()
            names = gateway.choose(task)
            answer, trace = await mcp_agent(task, gateway)
            rows.append({"task": key, "ok": bool(check(answer)), "answer": answer,
                         "tools": len(names), "found": found(key, names),
                         "input_tokens": sum(s["input_tokens"] for s in trace),
                         "calls": [(c["name"], c["arguments"]) for s in trace
                                   for c in s["calls"]]})
    r = {"ok": sum(w["ok"] for w in rows), "tools": rows[0]["tools"],
         "tasks_with_their_tools": sum(w["found"] for w in rows),
         "input_tokens": sum(w["input_tokens"] for w in rows), "rows": rows}
    return r

In [ ]:
clinic_and_co = [clinic, pharmacy, billing]
others = [create_service(n, t) for n, t in OTHER_SERVICES.items()]
everything = clinic_and_co + others
results = {}
for name, servers, selection in [
        ("Only the clinic's three servers", clinic_and_co, None),
        ("All the servers", everything, None),
        ("All, with the 10 most similar to the task", everything, 10),
        ("All, with the 5 most similar to the task", everything, 5)]:
    results[name] = r = await solve(servers, selection)
    print(f"{name:44s}", {k: v for k, v in r.items() if k != "rows"})

## Exercises

1. Explain in your own words what the gateway does that chapter 4's host did not do. What does the clinic gain from a single entry point, besides the selection?

2. Count the tokens of the schemas of the 56 tools and of the 13 tools of the clinic's three servers. How much does it cost, in each step, to see the twelve servers?

3. Without the language model, measure in how many of the sixteen tasks the tools that are needed are present if the gateway chooses 3, 5, 7, 10 and 15. From what number does it find them in all of them?

4. Look at the tasks in which the selection of 5 fails. Which tool is left out, and why is it not very similar to the task? Improve its description and repeat exercise 3.

5. Make the gateway, instead of the k most similar tools, give the model all the tools of the server of the most similar tool. Does it find more tasks? How many tools does the model see?

6. Add to the gateway a list of tools by profile: the front desk cannot use the accounting or staff tools. Where is it checked, when choosing or when calling? Why is it worth doing it in both places?

7. Using the calls stored in `results.json`, count how many each server receives in each variant. Does any reach the nine new services?

8. Write, for the pharmacy server, the entry that would describe it in an MCP server registry: name, description, version and how it is launched. What data does a host need to connect to it without reading its code?

9. The clinic's agent could talk to the lab's agent instead of its tools. Write the card that the lab's agent would publish with the A2A protocol: what it can do and how to call it. What changes compared with an MCP server?

10. **Project.** Put your server from chapter 4 together with the clinic's twelve behind the gateway, and measure whether the selection finds the tools of your three tasks.

## References

Model [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) and embedding model [multilingual-e5-small](https://huggingface.co/intfloat/multilingual-e5-small). [MCP server registry](https://github.com/modelcontextprotocol/registry). [A2A protocol](https://a2a-protocol.org). Northside Veterinary Clinic, its services and its data are fictional. Original code and experiments.